# Session 2 — Group Challenge

## From one indicator to four — then prove what you collected

All groups of **two or three** complete the same exercise.

**Fixed scope**

- Source: World Bank Indicators API V2
- Countries: Spain (`ESP`), Poland (`POL`), Morocco (`MAR`), Türkiye (`TUR`)
- Period: 2015–2024
- Grain: one row per **country × year × indicator**

**Phase 1** — Generalise the working one-indicator collection to four indicators.  
**Phase 2** — Find and implement the largest useful set of validation controls.  
**Phase 3** — Present your controls and identify your four strongest pieces of evidence.

Do not optimise for the most code. Optimise for evidence that would reveal a wrong, incomplete or misinterpreted collection.


## Target indicators

| Code | Indicator | Documented unit |
|---|---|---|
| `SP.POP.TOTL` | Population, total | people |
| `NY.GDP.PCAP.CD` | GDP per capita (current US$) | current US dollars per person |
| `NY.GDP.MKTP.KD.ZG` | GDP growth (annual %) | annual percent |
| `IT.NET.USER.ZS` | Individuals using the Internet (% of population) | percent of population |

The generic `unit` field in the observations may be empty. The indicator definition remains part of the data contract.


## 0. Imports and fixed scope


In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)


In [2]:
# Fixed exercise scope: do not change it while comparing groups.
COUNTRIES = ["ESP", "POL", "MAR", "TUR"]
COUNTRY_PATH = ";".join(COUNTRIES)
START_YEAR = 2015
END_YEAR = 2024
EXPECTED_YEARS = list(range(START_YEAR, END_YEAR + 1))
EXPECTED_ROWS_PER_INDICATOR = len(COUNTRIES) * len(EXPECTED_YEARS)

BASE_URL = "https://api.worldbank.org/v2/country/{countries}/indicator/{indicator}"
PARAMS = {
    "format": "json",
    "date": f"{START_YEAR}:{END_YEAR}",
    "per_page": 100,
}

# The classroom demonstration starts with this one indicator.
DEMO_INDICATOR = "SP.POP.TOTL"

# Normal classroom mode calls the live API. For a network outage, set the
# environment variable S2_USE_LOCAL_SNAPSHOTS=1 before running the notebook.
USE_LOCAL_SNAPSHOTS = os.getenv("S2_USE_LOCAL_SNAPSHOTS", "0") == "1"
DATA_DIR = Path("../Data/fallback")
OUTPUT_DIR = Path("../Outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Countries:", COUNTRIES)
print("Years:", f"{START_YEAR}–{END_YEAR}")
print("Expected rows per indicator:", EXPECTED_ROWS_PER_INDICATOR)
print("Source mode:", "local snapshots" if USE_LOCAL_SNAPSHOTS else "live API")


Countries: ['ESP', 'POL', 'MAR', 'TUR']
Years: 2015–2024
Expected rows per indicator: 40
Source mode: local snapshots


## 1. Working baseline — one indicator

Run this section before changing anything. It reproduces the population request used in the first part of the session.


In [3]:
SNAPSHOT_FILES = {
    "SP.POP.TOTL": "world_bank_population_total_2015_2024.json",
    "NY.GDP.PCAP.CD": "world_bank_gdp_per_capita_current_usd_2015_2024.json",
    "NY.GDP.MKTP.KD.ZG": "world_bank_gdp_growth_annual_pct_2015_2024.json",
    "IT.NET.USER.ZS": "world_bank_internet_users_pct_2015_2024.json",
}


def collect_one_indicator(indicator_code: str) -> dict:
    """Collect one World Bank response using one simple GET request."""
    endpoint = BASE_URL.format(countries=COUNTRY_PATH, indicator=indicator_code)
    prepared_url = requests.Request("GET", endpoint, params=PARAMS).prepare().url
    collected_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

    if USE_LOCAL_SNAPSHOTS:
        # Manual classroom fallback: same raw JSON shape as the live API.
        payload = json.loads((DATA_DIR / SNAPSHOT_FILES[indicator_code]).read_text(encoding="utf-8"))
        status_code = None
        called_url = prepared_url
        source_mode = "local snapshot"
    else:
        response = requests.get(endpoint, params=PARAMS, timeout=10)
        status_code = response.status_code
        called_url = response.url
        response.raise_for_status()
        payload = response.json()
        source_mode = "live API"

    return {
        "indicator_code": indicator_code,
        "endpoint": endpoint,
        "parameters": PARAMS.copy(),
        "called_url": called_url,
        "status_code": status_code,
        "source_mode": source_mode,
        "collected_at_utc": collected_at,
        "payload": payload,
    }


def observations_to_dataframe(payload: list) -> pd.DataFrame:
    """Transform payload[1] to one row per country × year × indicator."""
    observations = payload[1]
    frame = pd.json_normalize(observations).rename(
        columns={
            "indicator.id": "indicator_code",
            "indicator.value": "indicator_name",
            "countryiso3code": "country_code",
            "country.value": "country_name",
            "date": "year",
        }
    )
    columns = [
        "indicator_code", "indicator_name", "country_code", "country_name",
        "year", "value", "unit", "obs_status", "decimal",
    ]
    frame = frame[columns].copy()
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    return frame.sort_values(["indicator_code", "country_code", "year"]).reset_index(drop=True)


In [4]:
population_result = collect_one_indicator(DEMO_INDICATOR)

print("HTTP status:", population_result["status_code"])
print("Called URL:", population_result["called_url"])
print("Top-level type:", type(population_result["payload"]).__name__)
print("Top-level length:", len(population_result["payload"]))


HTTP status: None
Called URL: https://api.worldbank.org/v2/country/ESP;POL;MAR;TUR/indicator/SP.POP.TOTL?format=json&date=2015%3A2024&per_page=100
Top-level type: list
Top-level length: 2


In [5]:
population_metadata, population_observations = population_result["payload"]

display(population_metadata)
display(population_observations[0])
print("Raw observations:", len(population_observations))


{'page': 1, 'pages': 1, 'per_page': 100, 'total': 40, 'sourceid': '2', 'lastupdated': '2026-07-13'}
{'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'}, 'country': {'id': 'ES', 'value': 'Spain'}, 'countryiso3code': 'ESP', 'date': '2024', 'value': 48848840, 'unit': '', 'obs_status': '', 'decimal': 0}
Raw observations: 40


In [6]:
population_df = observations_to_dataframe(population_result["payload"])
display(population_df.head())
print("Population rows:", len(population_df))


  indicator_code     indicator_name country_code country_name  year     value  \
0    SP.POP.TOTL  Population, total          ESP        Spain  2015  46422303   
1    SP.POP.TOTL  Population, total          ESP        Spain  2016  46458139   
2    SP.POP.TOTL  Population, total          ESP        Spain  2017  46571232   
3    SP.POP.TOTL  Population, total          ESP        Spain  2018  46782011   
4    SP.POP.TOTL  Population, total          ESP        Spain  2019  47118501   

  unit obs_status  decimal  
0                        0  
1                        0  
2                        0  
3                        0  
4                        0  
Population rows: 40


## 2. Phase 1 — generalise from one indicator to four

Modify **only the configuration below** first. Add the three missing indicator codes from the table above. The loop already reuses the same collector and transformation.

Expected result after your modification: **160 rows** = 4 indicators × 4 countries × 10 years.


In [7]:
INDICATORS_TO_COLLECT = {
    "SP.POP.TOTL": "Population, total",
    # TODO — add GDP per capita
    # TODO — add GDP growth
    # TODO — add Internet use
}

collection_results = {}
frames = []

for indicator_code in INDICATORS_TO_COLLECT:
    result = collect_one_indicator(indicator_code)
    collection_results[indicator_code] = result
    frames.append(observations_to_dataframe(result["payload"]))

all_data = pd.concat(frames, ignore_index=True)

display(all_data.head())
print("Indicators collected:", all_data["indicator_code"].nunique())
print("Rows collected:", len(all_data))


  indicator_code     indicator_name country_code country_name  year     value  \
0    SP.POP.TOTL  Population, total          ESP        Spain  2015  46422303   
1    SP.POP.TOTL  Population, total          ESP        Spain  2016  46458139   
2    SP.POP.TOTL  Population, total          ESP        Spain  2017  46571232   
3    SP.POP.TOTL  Population, total          ESP        Spain  2018  46782011   
4    SP.POP.TOTL  Population, total          ESP        Spain  2019  47118501   

  unit obs_status  decimal  
0                        0  
1                        0  
2                        0  
3                        0  
4                        0  
Indicators collected: 1
Rows collected: 40


### Phase 1 checkpoint

Record your evidence before moving on.

- Indicators requested: …
- Indicators returned: …
- Expected rows: …
- Observed rows: …
- One row represents: …
- Any discrepancy: …


## 3. Phase 2 — build the broadest useful control set

Your goal is to detect as many distinct failure modes as possible. Do not count two differently worded versions of the same check as two controls.

For every control, record:

1. the question it answers;
2. the Python evidence;
3. the observed result;
4. the expected result;
5. a verdict (`PASS`, `FAIL`, `WARNING` or `REVIEW`);
6. its strength (`strong`, `medium` or `weak`) and why.

A large number of rows is not automatically strong evidence. Try to include independent checks of the request, response, structure, scope, keys, values, definitions and provenance.


In [8]:
control_results = []


def record_control(category, control, observed, expected, verdict, strength, rationale):
    """Store one control in a presentation-ready format."""
    control_results.append(
        {
            "category": category,
            "control": control,
            "observed": observed,
            "expected": expected,
            "verdict": verdict,
            "strength": strength,
            "rationale": rationale,
        }
    )


# TODO — implement your controls below.
# Example structure only (replace the placeholders):
# record_control(
#     category="...",
#     control="...",
#     observed=...,
#     expected="...",
#     verdict="PASS",
#     strength="strong",
#     rationale="...",
# )


In [9]:
control_report = pd.DataFrame(control_results)
display(control_report)
print("Number of distinct controls:", len(control_report))


Empty DataFrame
Columns: []
Index: []
Number of distinct controls: 0


## 4. Phase 3 — prepare the group presentation

You have **four minutes**. Present:

- the modification that produced the four-indicator DataFrame;
- the total number of distinct controls implemented;
- your **four strongest** controls, with observed evidence;
- one weak or misleading control and why it is insufficient;
- your verdict: **CERTIFIED** or **NOT YET CERTIFIED** for the declared scope.

### Group notes

**Total controls:** …  
**Four strongest controls:** …  
**Weak control:** …  
**Verdict:** …  
**Reason:** …


## 5. AI audit — complete individually

- **AI tool used:** …
- **Main prompt:** …
- **One AI proposal you verified:** …
- **How you verified it:** …
- **One AI proposal you corrected or rejected:** …
- **Why:** …

> AI can generate an extraction. The analyst must generate the evidence.
